# Session 9 — Regular Expressions & Text Cleaning

> raw strings, search/fullmatch/findall/sub, groups — then flags, compile, VERBOSE, lazy matching, sub with a function.

Read each cell, **predict** the output, then run it with **Shift + Enter**. The practice (solutions collapsed) is at the end.

**Tips:** press **Tab** to autocomplete a name, **Shift + Tab** for a function's help. Need a library? Run `%pip install <name>` in a cell (e.g. `%pip install pandas`) — in the browser (JupyterLite) it fetches a Pyodide build and lasts for the session.

In [ ]:
import re

In [ ]:
# --- 1. Validate: does the whole string match the pattern? ---------------
def valid_university_email(addr: str) -> bool:
    # \w+ chars, @, domain, literal dot (\.), then "edu"
    return re.fullmatch(r"\w+@\w+\.edu", addr) is not None

for addr in ["ana@university.edu", "ana@gmail.com", "ana@@x.edu"]:
    print(f"{addr:22} -> {valid_university_email(addr)}")

In [ ]:
# --- 2. Search anywhere; extract pieces with capture groups -------------
m = re.search(r"([A-Z]{2})(\d{4})", "Course ED1234 meets on Tue")
print("\ndept:", m.group(1), "| number:", m.group(2), "| whole:", m.group(0))

In [ ]:
# --- 3. NAMED groups read better than group(1)/group(2) -----------------
m = re.search(r"(?P<dept>[A-Z]{2})(?P<num>\d{4})", "ED1234")
print("named:", m.group("dept"), m.group("num"), "| dict:", m.groupdict())

In [ ]:
# --- 4. re.compile + re.VERBOSE: a reusable, self-documenting pattern ----
COURSE = re.compile(r"""
    (?P<dept>[A-Z]{2})    # two-letter department
    (?P<num>\d{4})        # four-digit course number
""", re.VERBOSE)
print("compiled findall:", COURSE.findall("Take ED1234 and PS5500 this term"))

In [ ]:
# --- 5. The walrus := keeps the match object without a second lookup -----
if (hit := re.search(r"\d+", "cohort 2026 has 30 students")):
    print("first number found:", hit.group())     # 2026

In [ ]:
# --- 6. The "." trap -----------------------------------------------------
print('\n"." matches ANY char:', re.search(r".", "a.b").group())   # 'a', not '.'
print('literal dot with \\.:   ', re.search(r"\.", "a.b").group())  # '.'

In [ ]:
# --- 7. Substitute / split / clean text ---------------------------------
messy = "  too    much\t  space  "
print("\ncleaned:", repr(re.sub(r"\s+", " ", messy).strip()))
print("re.split:", re.split(r"\s*,\s*", "a , b,c ,  d"))   # split on commas + stray spaces

In [ ]:
# --- 8. Mine free-text survey responses ---------------------------------
responses = ["loved #python and #stats", "more #python please", "no tags here"]
tags = []
for r in responses:
    tags += re.findall(r"#(\w+)", r)      # findall returns all matches
print("\nall tags:", tags)
from collections import Counter
print("tag counts:", Counter(tags))

In [ ]:
# --- 9. Reformat "Last, First" -> "First Last" ---------------------------
def flip_name(s: str) -> str:
    m = re.search(r"^(.+),\s*(.+)$", s.strip())
    return f"{m.group(2)} {m.group(1)}" if m else s

print("\n", flip_name("Curie, Marie"))      # Marie Curie

In [ ]:
# --- 10. When NOT to use regex ------------------------------------------
# For simple splits/trims, string methods are clearer than regex:
print("\nuse .split():", "a,b,c".split(","))
print("use .strip():", "  hi  ".strip())

In [ ]:
# ==========================================================================
# GOING DEEPER — second-hour material

In [ ]:
# ==========================================================================

In [ ]:
# --- deeper 1: flags -----------------------------------------------------------------

In [ ]:
print("\n=== GOING DEEPER ===")
corpus = "Stress was high. Some stress is normal. STRESS!"
print("ignorecase count:", len(re.findall(r"stress", corpus, re.IGNORECASE)))
lines = "42 Ana\n7 Ben\nnope Cara"
print("per-line numbers:", re.findall(r"^\d+", lines, re.MULTILINE))

In [ ]:
# --- deeper 2: greedy vs lazy ---------------------------------------------------------
tags = "[ED101][ED102]"
print("greedy .+ :", re.search(r"\[(.+)\]", tags).group(1))
print("lazy  .+? :", re.search(r"\[(.+?)\]", tags).group(1))

In [ ]:
# --- deeper 3: re.sub with a FUNCTION — the anonymizer ---------------------------------
transcript = "Ana said the course helped. Ben disagreed. Ana insisted."
ids = {}
def anonymize(m):
    name = m.group(0)
    ids.setdefault(name, f"P{len(ids) + 1:03d}")
    return ids[name]

print(re.sub(r"\b(?:Ana|Ben|Cara)\b", anonymize, transcript))
print("mapping:", ids)

## Now you try

The **In class** tasks first, **In class — going deeper** in the second hour, and **Homework** before the next session.

## In class

Always use raw strings `r"..."`. Predict each result before running.

### Task 1 — Validate
Write `valid_university_email(addr)` returning `True` only for `something@something.edu`.
Test: `"ana@university.edu"`, `"ana@gmail.com"`, `"a@b.edu.evil.com"`.

### Task 2 — Extract with groups
From `"Course ED1234 meets Tue"`, pull the department (`ED`) and number (`1234`) using one
regex with two capture groups.

### Task 3 — Clean
Collapse all runs of whitespace in `"  too    much\t space "` to single spaces and trim.

### Task 4 — Mine free text
From a list of open-ended responses, count how often each `#hashtag` appears
(use `re.findall(r"#(\w+)", text)` and `collections.Counter`).

### Task 5 — Reformat
Turn `"Curie, Marie"` into `"Marie Curie"` with a single regex + groups.

### Task 6 — Judgment
Give one task where a plain string method (`.split()`, `.strip()`, `.replace()`) is the better,
clearer choice than a regex.

### Bonus — Pythonic idiom drill
Cover the `# ->` answers, predict each line, then run.

```python
import re
m = re.search(r"(?P<year>\d{4})", "class of 2026")
print(m.group("year"), m.groupdict())   # -> 2026 {'year': '2026'}   (named groups)
print(re.split(r"\s*,\s*", "a, b ,c")) # -> ['a', 'b', 'c']        (split on commas + spaces)
```

## In class — going deeper (second hour)

### E1 — Named groups
Redo the dept+number extraction with `(?P<dept>...)` / `(?P<num>...)` and print
`m.groupdict()`.

### E2 — regex101 field trip
Paste your email pattern into regex101.com (flavor: **Python**). Read the left-panel
explanation token by token — does it say what you *meant*?

### D1 — A commented pattern
Rewrite your email validator with `re.compile(..., re.VERBOSE)`, one commented line per
token. It must behave identically.

### D2 — The anonymizer
Replace every occurrence of the names Ana/Ben/Cara in a transcript with stable IDs
`P001`, `P002`, … (same name → same ID) using `re.sub` with a **function** replacement.

### D3 — Two-format date harvest
From `"submitted 2026-07-06, revised 07/08/2026, due 2026-09-01"`, extract **all** dates
in either `YYYY-MM-DD` or `MM/DD/YYYY` form with one `findall` and an alternation.

### D4 — Case-insensitive keyword count
Count mentions of "stress" in a paragraph regardless of case, with a flag — not by
lowercasing the text.

## Homework (before Session 10)

*~30–45 minutes, outside class — it doesn't count toward class time. Try everything before peeking at the solutions.*

### H1 — Pattern drill
One `re.fullmatch` pattern each; test against two valid and two invalid strings:
1. Student ID: two uppercase letters + six digits (`AB123456`)
2. US phone: `555-867-5309` (digits and dashes only)
3. ISO date: `2026-07-06` (just the shape — don't validate month ranges)

### H2 — Messy-name cleanup
Normalize `["  smith,  ana", "LEE,BEN", "Garcia ,  Cara "]` to `"Ana Smith"`,
`"Ben Lee"`, `"Cara Garcia"`: regex-split on the comma (with optional spaces around it),
then `.strip()` + `.title()`, and flip the order.

### H3 — Domain harvest
From a paragraph containing several email addresses, extract the **unique** domains
(e.g. `{"university.edu", "gmail.com"}`) with one `findall` + a `set`.

In [ ]:
# Your practice work — type here. Predict before you run.


<details>
<summary><strong>Show solutions</strong></summary>

### In class

See `demo.py` in this folder — it implements all six. Key lines:

```python
re.fullmatch(r"\w+@\w+\.edu", addr) is not None      # 1 (fullmatch anchors both ends)
m = re.search(r"([A-Z]{2})(\d{4})", s); m.group(1), m.group(2)   # 2
re.sub(r"\s+", " ", messy).strip()                   # 3
from collections import Counter; Counter(re.findall(r"#(\w+)", text))   # 4
m = re.search(r"^(.+),\s*(.+)$", s); f"{m.group(2)} {m.group(1)}"      # 5
```
Task 6: splitting `"a,b,c"` on commas is just `"a,b,c".split(",")` — no regex needed.
Reach for regex only when the pattern is genuinely variable (digits, optional parts, anchors).

Trap reminder: `.` matches **any** character — use `\.` for a literal dot, and never forget the
`r"..."` prefix or your backslashes become Python escape sequences.

### In class — going deeper

```python
# E1
import re
m = re.search(r"(?P<dept>[A-Z]{2})(?P<num>\d{4})", "Course ED1234 meets Tue")
print(m.group("dept"), m.group("num"))   # ED 1234
print(m.groupdict())                     # {'dept': 'ED', 'num': '1234'}
```

E2 — the point is the habit: regex101's explainer catches "`.` matches any character"
mistakes before your data does.

```python
import re

# D1
EMAIL = re.compile(r"""
    \w+          # the user part
    @
    \w+          # the domain
    \.edu        # a literal dot, then edu
""", re.VERBOSE)
print(EMAIL.fullmatch("ana@university.edu") is not None)   # True

# D2
ids = {}
def anonymize(m):
    name = m.group(0)
    ids.setdefault(name, f"P{len(ids) + 1:03d}")
    return ids[name]

text = "Ana said X. Ben said Y. Ana agreed."
print(re.sub(r"\b(?:Ana|Ben|Cara)\b", anonymize, text))
# P001 said X. P002 said Y. P001 agreed.

# D3
s = "submitted 2026-07-06, revised 07/08/2026, due 2026-09-01"
print(re.findall(r"\d{4}-\d{2}-\d{2}|\d{2}/\d{2}/\d{4}", s))
# ['2026-07-06', '07/08/2026', '2026-09-01']

# D4
para = "Stress was high. Some stress is normal. STRESS!"
print(len(re.findall(r"stress", para, re.IGNORECASE)))     # 3
```

### Homework

```python
# H1
import re

sid   = r"[A-Z]{2}\d{6}"        # AB123456 ✓  CD000001 ✓  ab123456 ✗  AB12345 ✗
phone = r"\d{3}-\d{3}-\d{4}"    # 555-867-5309 ✓  555-8675309 ✗
date  = r"\d{4}-\d{2}-\d{2}"    # 2026-07-06 ✓  2026-7-6 ✗

for s in ["AB123456", "ab123456"]:
    print(s, re.fullmatch(sid, s) is not None)

# H2
def clean_name(raw):
    last, first = re.split(r"\s*,\s*", raw.strip(), maxsplit=1)
    return f"{first.strip().title()} {last.strip().title()}"

for raw in ["  smith,  ana", "LEE,BEN", "Garcia ,  Cara "]:
    print(clean_name(raw))      # Ana Smith · Ben Lee · Cara Garcia

# H3
text = "Write ana@university.edu or ben@gmail.com; cc cara@university.edu today."
print(set(re.findall(r"\w+@([\w.]+\.\w+)", text)))
# {'university.edu', 'gmail.com'}  — the set removes the duplicate
```

</details>